# Code review: ревью чужого кода и работа по замечаниям

**Учебное задание (2 ч).** Среда — Google Colab, язык — Java (каждый пример — один файл, запуск `java Файл.java`). Работа парная: вторая часть выполняется на репозитории однокурсника.

Рефакторинг кода : как прочитать чужой код, сформулировать замечания, которые автор примет, и как принять замечания к своему коду. Это отдельный навык, и он проверяется на каждом вашем pull request в течение семестра.

**Что получится на выходе**

| Артефакт | Где |
|---|---|
| Таблица замечаний к выданному фрагменту | `ReviewNotes.csv` |
| Исправленный по замечаниям код с тестами | `ShiftPlanFixed.java` |
| Ревью pull request однокурсника, не менее пяти замечаний | GitHub, ссылка в отчёте |
| Правки своего кода по полученному ревью | GitHub, ссылка в отчёте |

Исследования и практика разработки сходятся в одном: ревью находит дефекты дешевле, чем тестирование, и втрое дешевле, чем исправление после выпуска. Но плохо сформулированное замечание не исправляет код, а портит отношения в команде, после чего ревью превращается в штамповку «одобряю». Поэтому формулировка здесь — предмет изучения наравне с содержанием.

---
## 1. Подготовка

In [ ]:
!java -version

[0.001s][warning][os,container] Cgroup memory controller path at '/sys/fs/cgroup' seems to have moved to '/../../jupyter-children', detected limits won't be accurate
[0.001s][warning][os,container] Cgroup cpu controller path at '/sys/fs/cgroup' seems to have moved to '/../../jupyter-children', detected limits won't be accurate
openjdk version "21.0.12.1" 2026-08-18
OpenJDK Runtime Environment (build 21.0.12.1+1-1-24.04.4-Ubuntu)
OpenJDK 64-Bit Server VM (build 21.0.12.1+1-1-24.04.4-Ubuntu, mixed mode, sharing)


In [ ]:
import csv, os, re, subprocess, datetime
import pandas as pd

os.makedirs("ревью", exist_ok=True)
print("Среда готова.")

Среда готова.


---
## 2. Чек-лист ревью

Ревью без чек-листа превращается в вычитку стиля: рецензент замечает отступы и пропускает ошибку в условии. Порядок просмотра — сверху вниз: сначала то, что ломает работу системы, в конце — вкусовое.

| Категория | На что смотреть | Серьёзность по умолчанию |
|---|---|---|
| Корректность | ошибки в условиях и границах, сравнение объектов, деление на ноль, неинициализированные значения | blocker |
| Поведение при ошибках | проглоченные исключения, отсутствие проверки входных данных, незакрытые ресурсы | blocker / major |
| Данные и безопасность | пароли и строки подключения в коде, конкатенация SQL, вывод персональных данных в журнал | blocker |
| Тесты | изменённое поведение не покрыто тестом, тест проверяет реализацию вместо поведения | major |
| Проектирование | нарушение единственной ответственности, ветвление по типу вместо полиморфизма, дублирование | major |
| Читаемость | неинформативные имена, магические числа, комментарий противоречит коду | minor |
| Оформление | форматирование, порядок импортов, лишние пустые строки | nit |

**Шкала серьёзности.** `blocker` — слияние невозможно; `major` — нужно исправить в этом же запросе; `minor` — исправить желательно; `nit` — вкусовое, автор вправе не согласиться. Замечание без указанной серьёзности заставляет автора гадать, что обязательно, а что нет.

### Как формулируется замечание

Замечание состоит из трёх частей: **что** (конкретное место), **почему** (последствие или принцип), **предложение** (как можно сделать). Говорят о коде, а не об авторе.

| Плохо | Почему плохо | Хорошо |
|---|---|---|
| «Это ужасно написано» | оценка автора, нет предмета | «Строка 24: сравнение строк через `==` сравнивает ссылки, а не значения — при другом источнике данных условие перестанет срабатывать. Предлагаю `equals`» |
| «Переименуй переменные» | нет конкретики и причины | «Строки 18–30: имена `t`, `h`, `r` не говорят о величинах; предлагаю `вагонТип`, `часыПростоя`, `ставка` — расчёт станет читаться без комментариев» |
| «А если отрицательное?» | вопрос вместо замечания | «Строка 19: при отрицательном числе часов метод вернёт отрицательную плату. Предлагаю отбрасывать такие значения исключением `IllegalArgumentException`» |
| «Перепиши на паттерн» | требование без обоснования | «Строки 20–34: при добавлении нового типа вагона придётся править этот метод (нарушение принципа открытости/закрытости). Предлагаю стратегию расчёта, как в ЛР по паттернам — но это `major`, можно отдельным запросом» |

---
## 3. Образец: разбор выданного фрагмента

Ниже — метод расчёта платы за простой вагонов. Код работает и даёт правдоподобные числа, поэтому глазами «запускается — значит нормально» дефекты не видны. Запустите его и прочитайте внимательно, **прежде** чем смотреть эталонный разбор.

In [ ]:
%%writefile DemurrageLegacy.java
public class DemurrageLegacy {

    public static void main(String[] args) {
        calc("крытый", 30, 2.5);
        calc("цистерна", 30, 2.5);
        calc("платформа", -5, 2.5);
    }

    // Расчет платы за простой вагона сверх нормы. Норма - одни сутки.
    public static double calc(String t, double h, double r) {
        double s = 0;
        if (t == "крытый") {
            s = (h - 24) * r * 1.0;
        } else if (t == "цистерна") {
            s = (h - 24) * r * 1.15;
        } else {
            s = (h - 24) * r * 1.0;
        }
        if (s > 1000) {
            s = s * 0.9;   // скидка
        }
        System.out.println("Плата: " + s);
        return s;
    }
}

Overwriting DemurrageLegacy.java


In [ ]:
!java DemurrageLegacy.java

[0.000s][warning][os,container] Cgroup memory controller path at '/sys/fs/cgroup' seems to have moved to '/../../jupyter-children', detected limits won't be accurate
[0.000s][warning][os,container] Cgroup cpu controller path at '/sys/fs/cgroup' seems to have moved to '/../../jupyter-children', detected limits won't be accurate
Плата: 15.0
Плата: 17.25
Плата: -72.5


### 3.1 Эталонная таблица замечаний

Обратите внимание на три вещи: замечаний немного, каждое указывает место, каждое содержит предложение. Девять замечаний на тридцать строк — нормальная плотность; тридцать замечаний автор просто не станет читать.

In [ ]:
эталонные_замечания = [
    dict(Строка="22", Категория="Корректность", Серьёзность="blocker",
         Замечание="Сравнение строк через == сравнивает ссылки. Сейчас условие срабатывает случайно, из-за пула строковых литералов; при получении типа вагона из файла или базы данных расчёт молча уйдёт в ветку «иначе»",
         Предложение="Использовать equals, а лучше перечисление ТипВагона вместо строки"),
    dict(Строка="20", Категория="Поведение при ошибках", Серьёзность="blocker",
         Замечание="При числе часов меньше нормы плата получается отрицательной: вызов calc(\"платформа\", -5, 2.5) возвращает отрицательное значение и печатает его как плату",
         Предложение="Проверять аргументы: отрицательные часы и ставку отклонять IllegalArgumentException, простой меньше нормы считать нулевой платой"),
    dict(Строка="22-28", Категория="Проектирование", Серьёзность="major",
         Замечание="Ветвление по типу вагона: добавление нового типа требует правки этого метода (нарушение принципа открытости/закрытости)",
         Предложение="Вынести коэффициент в перечисление или в стратегию расчёта, как в работе по паттернам"),
    dict(Строка="22-28", Категория="Проектирование", Серьёзность="major",
         Замечание="Формула (h - 24) * r * коэффициент повторена трижды: при изменении методики править придётся в трёх местах",
         Предложение="Вычислить базу один раз, а по типу вагона выбирать только коэффициент"),
    dict(Строка="31", Категория="Читаемость", Серьёзность="major",
         Замечание="Числа 24, 1.15, 1000 и 0.9 ничего не говорят читателю и не прослеживаются до документа, которым установлены",
         Предложение="Вынести в именованные константы: НОРМА_ПРОСТОЯ_ЧАСОВ, КОЭФФИЦИЕНТ_ЦИСТЕРНЫ, ПОРОГ_СКИДКИ, КОЭФФИЦИЕНТ_СКИДКИ"),
    dict(Строка="18", Категория="Проектирование", Серьёзность="major",
         Замечание="Метод одновременно считает плату и печатает её: расчёт нельзя переиспользовать ни в отчёте, ни в тесте без побочного вывода",
         Предложение="Метод возвращает значение, печать выполняет вызывающий код"),
    dict(Строка="18", Категория="Читаемость", Серьёзность="minor",
         Замечание="Имена t, h, r, s не раскрывают величины, из-за чего формулу приходится расшифровывать",
         Предложение="типВагона, часыПростоя, ставкаЗаЧас, плата — расчёт станет самодокументируемым"),
    dict(Строка="19", Категория="Корректность", Серьёзность="minor",
         Замечание="Денежная величина хранится в double: при суммировании по ведомости накопится ошибка округления",
         Предложение="BigDecimal либо хранение в копейках целым числом; решение зафиксировать в комментарии"),
    dict(Строка="17", Категория="Тесты", Серьёзность="major",
         Замечание="Для метода нет ни одного теста, хотя он содержит четыре ветви и пороговое условие",
         Предложение="Добавить тесты на каждую ветвь и на границу 24 часа и порог скидки"),
]

эталон = pd.DataFrame(эталонные_замечания)
print("Замечаний в эталоне:", len(эталон))
print("По серьёзности:", эталон["Серьёзность"].value_counts().to_dict())
эталон[["Строка", "Категория", "Серьёзность", "Замечание"]]

Замечаний в эталоне: 9
По серьёзности: {'major': 5, 'blocker': 2, 'minor': 2}


,Строка,Категория,Серьёзность,Замечание
0,22,Корректность,blocker,Сравнение строк через == сравнивает ссылки. Се...
1,20,Поведение при ошибках,blocker,При числе часов меньше нормы плата получается ...
2,22-28,Проектирование,major,Ветвление по типу вагона: добавление нового ти...
3,22-28,Проектирование,major,Формула (h - 24) * r * коэффициент повторена т...
4,31,Читаемость,major,"Числа 24, 1.15, 1000 и 0.9 ничего не говорят ч..."
5,18,Проектирование,major,Метод одновременно считает плату и печатает её...
6,18,Читаемость,minor,"Имена t, h, r, s не раскрывают величины, из-за..."
7,19,Корректность,minor,Денежная величина хранится в double: при сумми...
8,17,Тесты,major,"Для метода нет ни одного теста, хотя он содерж..."


### 3.2 Чего в эталоне нет

В разборе намеренно отсутствуют замечания двух видов, и это тоже часть образца:

* **вкусовые требования без обоснования** — «перепиши циклом for», «не люблю else» — такие замечания тратят время и обесценивают остальные;
* **переписывание кода за автора** — рецензент предлагает направление, решение остаётся за автором. Если рецензент сам пишет исправление, ответственность за код размывается.

**Вопрос.** В эталоне нет замечания о том, что `calc` объявлен `static`. Должно ли оно быть? Обоснуйте.

**Ответ:** Нет, отдельного замечания про `static` здесь быть не должно. Метод не читает поля объекта, это функция трёх аргументов, и `static` для неё уместен. Требование «сделай метод экземплярным» без изменения поведения было бы вкусовым: рецензент переписывает форму, а не указывает дефект. Замечание появилось бы, если бы расчёт зависел от состояния объекта (коэффициент типа вагона как поле, стратегия) или если бы `static` держал общее изменяемое состояние. В образце рядом лежат настоящие дефекты: сравнение строк через `==`, отрицательная плата и печать внутри расчёта. Их и разобрали.


---
## 4. Задача 1: самостоятельный разбор

Второй фрагмент — формирование наряда локомотивных бригад на смену. Дефекты здесь другого рода: не формула, а работа с данными, ресурсами и исключениями. Запустите, изучите и составьте таблицу замечаний.

In [ ]:
%%writefile ShiftPlanLegacy.java
import java.io.FileWriter;
import java.text.SimpleDateFormat;
import java.util.ArrayList;
import java.util.Date;
import java.util.List;

public class ShiftPlanLegacy {

    static SimpleDateFormat dateFormat = new SimpleDateFormat("dd.MM.yyyy");
    static List<String> log = new ArrayList<>();

    public static void main(String[] args) {
        List<Brigade> brigades = new ArrayList<>();
        brigades.add(new Brigade("Иванов", 11.0, true));
        brigades.add(new Brigade("Петров", 0.0, true));
        brigades.add(new Brigade("Сидоров", 4.5, false));
        brigades.add(new Brigade("Козлов", 8.0, true));

        System.out.println(make(brigades, 3));
        System.out.println("Журнал: " + log);
    }

    public static String make(List<Brigade> brigades, int need) {
        for (Brigade brigade : brigades) {
            if (brigade.hoursPerShift > 10.0 || brigade.admitted == false) {
                brigades.remove(brigade);
            }
        }

        String result = "";
        for (int i = 0; i < need; i++) {
            result = result + brigades.get(i).lastName + "; ";
        }

        try {
            FileWriter writer = new FileWriter("наряд.txt");
            writer.write(result);
        } catch (Exception e) {
        }

        log.add(dateFormat.format(new Date()) + " наряд сформирован");
        return result;
    }
}

class Brigade {
    String lastName;
    double hoursPerShift;
    boolean admitted;

    Brigade(String lastName, double hoursPerShift, boolean admitted) {
        this.lastName = lastName;
        this.hoursPerShift = hoursPerShift;
        this.admitted = admitted;
    }
}

Overwriting ShiftPlanLegacy.java


In [ ]:
!java ShiftPlanLegacy.java

[0.000s][warning][os,container] Cgroup memory controller path at '/sys/fs/cgroup' seems to have moved to '/../../jupyter-children', detected limits won't be accurate
[0.000s][warning][os,container] Cgroup cpu controller path at '/sys/fs/cgroup' seems to have moved to '/../../jupyter-children', detected limits won't be accurate
Exception in thread "main" java.util.ConcurrentModificationException
	at java.base/java.util.ArrayList$Itr.checkForComodification(ArrayList.java:1095)
	at java.base/java.util.ArrayList$Itr.next(ArrayList.java:1049)
	at ShiftPlanLegacy.make(ShiftPlanLegacy.java:24)
	at ShiftPlanLegacy.main(ShiftPlanLegacy.java:19)


**Подсказка, куда смотреть.** Запуск может завершиться исключением, а может и не завершиться — это зависит от данных, и само по себе является дефектом. Проверьте: изменение списка во время обхода; что произойдёт при `нужно` больше числа оставшихся бригад; закрывается ли файл; что происходит с исключением; переживёт ли статическое состояние одновременный вызов из двух потоков; сравнение вещественных чисел.

### 4.1 Ваша таблица замечаний

Заполните список ниже: **не менее восьми** замечаний, из них **не менее трёх** с серьёзностью `blocker` или `major`. В каждом замечании обязательны место, последствие и предложение. Поле «Предложение» короче двадцати символов самопроверка не примет.

In [ ]:
# 4.1 Ваши замечания. Формат тот же, что в эталоне.
мои_замечания = [
    dict(
        Строка="23-27",
        Категория="Корректность",
        Серьёзность="blocker",
        Замечание="Удаление элементов из списка brigades прямо во время его обхода циклом for-each вызывает исключение ConcurrentModificationException (что и происходит при запуске программы)",
        Предложение="Использовать removeIf для безопасной фильтрации списка или собирать элементы для удаления в отдельный список"
    ),
    dict(
        Строка="30-33",
        Категория="Корректность",
        Серьёзность="blocker",
        Замечание="Если параметр need больше количества оставшихся после фильтрации бригад, метод выбросит IndexOutOfBoundsException",
        Предложение="Проверять размер отфильтрованного списка перед циклом и ограничивать итерацию либо возвращать ошибку"
    ),
    dict(
        Строка="35-39",
        Категория="Поведение при ошибках",
        Серьёзность="major",
        Замечание="Исключение при записи в файл перехватывается пустым блоком catch {}, из-за чего сбой ввода-вывода скрывается от вызывающего кода",
        Предложение="Обработать исключение должным образом или пробросить его выше (throws IOException), уведомив пользователя"
    ),
    dict(
        Строка="35",
        Категория="Поведение при ошибках",
        Серьёзность="major",
        Замечание="Ресурс FileWriter открывается без закрытия (нет блока try-with-resources), что приводит к утечке файловых дескрипторов",
        Предложение="Использовать конструкцию try-with-resources для автоматического закрытия FileWriter после записи"
    ),
    dict(
        Строка="11",
        Категория="Проектирование",
        Серьёзность="major",
        Замечание="Использование статической коллекции log и статического форматтера даты делает метод потоконебезопасным (не thread-safe)",
        Предложение="Убрать изменяемое состояние из статических полей: не хранить общий журнал в static, чтобы параллельные вызовы не портили друг другу данные"
    ),
    dict(
        Строка="24",
        Категория="Корректность",
        Серьёзность="minor",
        Замечание="Прямое сравнение вещественного числа с плавающей точкой через оператор больше (brigades.hoursPerShift > 10.0) может быть ненадежным из-за погрешности вычислений",
        Предложение="Учитывать допустимую погрешность (эпсилон) или использовать целые величины (например, минуты)"
    ),
    dict(
        Строка="10",
        Категория="Читаемость",
        Серьёзность="minor",
        Замечание="SimpleDateFormat не является потокобезопасным и объявлен статинчески, что может приводить к сбоям форматирования при многопоточности",
        Предложение="Использовать современный java.time.format.DateTimeFormatter или создавать экземпляр локально"
    ),
    dict(
        Строка="20",
        Категория="Тесты",
        Серьёзность="major",
        Замечание="Для метода make() отсутствуют автоматизированные юнит-тесты или проверки граничных условий",
        Предложение="Добавить набор тестов, проверяющих корректность формирования наряда при разных входных данных"
    )
]

ревью = pd.DataFrame(мои_замечания)
ревью.to_csv("ReviewNotes.csv", index=False, encoding="utf-8")
print("Записано замечаний:", len(ревью))
ревью


Записано замечаний: 8


,Строка,Категория,Серьёзность,Замечание,Предложение
0,23-27,Корректность,blocker,Удаление элементов из списка brigades прямо во...,Использовать removeIf для безопасной фильтраци...
1,30-33,Корректность,blocker,Если параметр need больше количества оставшихс...,Проверять размер отфильтрованного списка перед...
2,35-39,Поведение при ошибках,major,Исключение при записи в файл перехватывается п...,Обработать исключение должным образом или проб...
3,35,Поведение при ошибках,major,Ресурс FileWriter открывается без закрытия (не...,Использовать конструкцию try-with-resources дл...
4,11,Проектирование,major,Использование статической коллекции log и стат...,"Убрать состояние в статические поля, передават..."
5,24,Корректность,minor,Прямое сравнение вещественного числа с плавающ...,Учитывать допустимую погрешность (эпсилон) или...
6,10,Читаемость,minor,SimpleDateFormat не является потокобезопасным ...,Использовать современный java.time.format.Date...
7,20,Тесты,major,Для метода make() отсутствуют автоматизированн...,"Добавить набор тестов, проверяющих корректност..."


---
## 5. Задача 2: исправление по замечаниям

Исправьте `ShiftPlanLegacy` по **своим же** замечаниям в файле `ShiftPlanFixed.java`. Требования:

1. Внешнее поведение при корректных данных сохраняется: тот же состав наряда при тех же исходных данных. Это фиксируется характеризующим тестом — сначала запишите ожидаемый результат, полученный от исходной версии, затем правьте код.
2. Устранены все замечания серьёзности `blocker` и `major` из вашей таблицы.
3. Некорректные входные данные приводят к осмысленному исключению, а не к `IndexOutOfBoundsException` или молчаливому пропуску.
4. Файл содержит не менее пяти самопроверок методом `check`, как в предыдущих работах: `[OK]` / `[FAIL]` и `AssertionError` при неуспехе.
5. Первым в файле идёт открытый класс `ShiftPlanFixed` с методом `main`.

Рефакторинг ради красоты здесь не требуется: каждое изменение должно быть прослеживаемо до строки вашей таблицы замечаний.

In [ ]:
%%writefile ShiftPlanFixed.java
import java.io.FileWriter;
import java.io.IOException;
import java.util.ArrayList;
import java.util.List;
import java.util.stream.Collectors;

public class ShiftPlanFixed {

    private static final double MAX_HOURS_PER_SHIFT = 10.0;

    // Состав наряда для данных из main исходной программы при need = 2.
    // ShiftPlanLegacy на вызове make(..., 3) падает с ConcurrentModificationException
    // и строку наряда не печатает. Ожидание получено по тем же правилам отбора:
    // Иванов (11 ч) и Сидоров (не допущен) не входят, остаются Петров и Козлов.
    private static final String EXPECTED_PLAN = "Петров; Козлов; ";

    public static void main(String[] args) {
        runSelfChecks();
        System.out.println("Все самопроверки успешно пройдены!");
    }

    public static String make(List<BrigadeFixed> brigades, int need) {
        if (brigades == null) {
            throw new IllegalArgumentException("Список бригад не может быть null");
        }
        if (need < 0) {
            throw new IllegalArgumentException("Потребность в бригадах не может быть отрицательной");
        }

        List<BrigadeFixed> available = brigades.stream()
                .filter(b -> b.hoursPerShift <= MAX_HOURS_PER_SHIFT && b.admitted)
                .collect(Collectors.toList());

        if (need > available.size()) {
            throw new IllegalArgumentException(
                    "Требуемое число бригад превышает количество доступных допущенных бригад");
        }

        StringBuilder resultBuilder = new StringBuilder();
        for (int i = 0; i < need; i++) {
            resultBuilder.append(available.get(i).lastName).append("; ");
        }
        String result = resultBuilder.toString();

        try (FileWriter writer = new FileWriter("наряд.txt")) {
            writer.write(result);
        } catch (IOException e) {
            throw new IllegalStateException("Не удалось записать наряд в файл", e);
        }

        return result;
    }

    private static void check(String name, boolean ok) {
        if (ok) {
            System.out.println("[OK] " + name);
            return;
        }
        System.out.println("[FAIL] " + name);
        throw new AssertionError(name);
    }

    private static void runSelfChecks() {
        List<BrigadeFixed> list = new ArrayList<>();
        list.add(new BrigadeFixed("Иванов", 11.0, true));
        list.add(new BrigadeFixed("Петров", 0.0, true));
        list.add(new BrigadeFixed("Сидоров", 4.5, false));
        list.add(new BrigadeFixed("Козлов", 8.0, true));

        int sizeBefore = list.size();
        String res = make(list, 2);
        check("Самопроверка 1: наряд сформирован верно",
                EXPECTED_PLAN.equals(res) && list.size() == sizeBefore);

        boolean tooMany = false;
        try {
            make(list, 5);
        } catch (IllegalArgumentException e) {
            tooMany = true;
        }
        check("Самопроверка 2: корректно поймано ограничение размера", tooMany);

        boolean nullRejected = false;
        try {
            make(null, 1);
        } catch (IllegalArgumentException e) {
            nullRejected = true;
        }
        check("Самопроверка 3: корректная обработка null-аргумента", nullRejected);

        boolean negativeRejected = false;
        try {
            make(list, -1);
        } catch (IllegalArgumentException e) {
            negativeRejected = true;
        }
        check("Самопроверка 4: корректная обработка отрицательного значения need", negativeRejected);

        List<BrigadeFixed> strictList = new ArrayList<>();
        strictList.add(new BrigadeFixed("Алексеев", 5.0, true));
        check("Самопроверка 5: фильтрация отработала корректно",
                "Алексеев; ".equals(make(strictList, 1)));
    }
}

class BrigadeFixed {
    String lastName;
    double hoursPerShift;
    boolean admitted;

    BrigadeFixed(String lastName, double hoursPerShift, boolean admitted) {
        this.lastName = lastName;
        this.hoursPerShift = hoursPerShift;
        this.admitted = admitted;
    }
}


Writing ShiftPlanFixed.java


In [ ]:
!java ShiftPlanFixed.java

[0.000s][warning][os,container] Cgroup memory controller path at '/sys/fs/cgroup' seems to have moved to '/../../jupyter-children', detected limits won't be accurate
[0.000s][warning][os,container] Cgroup cpu controller path at '/sys/fs/cgroup' seems to have moved to '/../../jupyter-children', detected limits won't be accurate
[OK] Самопроверка 1: наряд сформирован верно
[OK] Самопроверка 2: корректно поймано ограничение размера
[OK] Самопроверка 3: корректная обработка null-аргумента
[OK] Самопроверка 4: корректная обработка отрицательного значения need
[OK] Самопроверка 5: фильтрация отработала корректно
Все самопроверки успешно пройдены!


**Таблица прослеживаемости.** Заполните соответствие «замечание → что изменено». Именно она, а не объём правок, показывает качество работы.

| № замечания | Серьёзность | Что изменено в коде |
|---|---|---|
| 1 | blocker | Удаление из `brigades` во время for-each заменено отбором в новый список `available` через `stream().filter`. Исходный список не меняется, `ConcurrentModificationException` больше не возникает. |
| 2 | blocker | Если `need` больше числа доступных бригад, метод выбрасывает `IllegalArgumentException`, а не `IndexOutOfBoundsException`. Так же отклоняются `null` и отрицательный `need`. |
| 3 | major | Пустой `catch` убран. Ошибка записи не проглатывается: она заворачивается в `IllegalStateException` и уходит вызывающему коду. |
| 4 | major | `FileWriter` открыт в try-with-resources и закрывается сам, в том числе когда запись падает. |
| 5 | major | Статические `log` и `SimpleDateFormat` удалены. У `make` больше нет общего изменяемого состояния между вызовами. |
| 6 | minor | Порог 10 часов вынесен в константу `MAX_HOURS_PER_SHIFT`. Сравнение с эпсилоном не добавлялось: в данных задания часы заданы литералами, для них значение точное, а серьёзность замечания — minor. |
| 7 | minor | `SimpleDateFormat` удалён вместе с журналом, общего небезопасного форматтера не осталось. |
| 8 | major | В `main` пять самопроверок через метод `check`: состав наряда и неизменность входного списка, слишком большой `need`, `null`, отрицательный `need`, одна допущенная бригада. |


---
## 6. Задача 3: ревью на GitHub

Эта часть выполняется в паре на действующих репозиториях курса.

**Как рецензент**

1. Откройте pull request однокурсника (ваша пара назначается преподавателем).
2. Начните ревью: вкладка *Files changed* → *Review changes* → *Start a review*. Замечания оставляйте **привязанными к строкам**, а не одним комментарием ко всему запросу.
3. Оставьте не менее пяти замечаний, из них хотя бы одно `blocker` или `major`. Серьёзность указывайте префиксом в тексте: `blocker:`, `major:`, `minor:`, `nit:`.
4. Завершите ревью с решением *Request changes* или *Comment*. Решение *Approve* при наличии незакрытых `blocker` недопустимо.
5. Отметьте в ревью хотя бы одно удачное решение автора. Ревью, состоящее только из претензий, перестают читать.

**Как автор**

6. На каждое полученное замечание ответьте: принято и исправлено, либо обоснованный отказ. Молчание ответом не считается.
7. Внесите правки отдельными коммитами со ссылкой на замечание.
8. Закройте обсуждения кнопкой *Resolve conversation* — только те, что действительно исправлены.

**Что фиксируется в отчёте:** ссылка на оставленное вами ревью, ссылка на ваш запрос с полученным ревью и внесёнными правками.

**В этой сдаче раздел не выполняется.** Парное ревью pull request однокурсника и правки своего кода по чужим замечаниям пропускаются. Ссылки ниже оставлены пустыми специально.


In [ ]:
# Парное ревью напарника в этой работе не выполняется.
# Замечания однокурснику не оставлялись, правок по чужому ревью нет.
ССЫЛКА_НА_МОЁ_РЕВЬЮ = ""      #@param {type:"string"}
ССЫЛКА_НА_МОЙ_PR = ""         #@param {type:"string"}
ЗАМЕЧАНИЙ_ОСТАВЛЕНО = 0       #@param {type:"integer"}
ЗАМЕЧАНИЙ_ПОЛУЧЕНО = 0        #@param {type:"integer"}
ЗАМЕЧАНИЙ_ПРИНЯТО = 0         #@param {type:"integer"}

print("Парное ревью пропущено.")
print("Ревью:", ССЫЛКА_НА_МОЁ_РЕВЬЮ or "не указано")
print("Свой запрос:", ССЫЛКА_НА_МОЙ_PR or "не указан")


---
## 7. Разбор и выводы

### 7.1 Сравнение своей таблицы с эталоном подхода

| Вопрос | Ответ |
|---|---|
| Сколько замечаний вы нашли и сколько из них `blocker`? | Восемь замечаний. Из них два `blocker`: удаление из списка во время обхода (`ConcurrentModificationException` на запуске) и обращение по индексу, когда `need` больше числа оставшихся бригад. Ещё четыре замечания — `major`: пустой `catch`, незакрытый `FileWriter`, общее статическое состояние и отсутствие тестов. |
| Какое замечание вы нашли не сразу, и что помешало увидеть его раньше? | Пустой `catch` и незакрытый файл. Запуск падает раньше, на `ConcurrentModificationException` в строке 24, до `FileWriter` выполнение не доходит. В выводе виден только первый дефект, остальные видны только при чтении кода ниже по методу. |
| Было ли среди ваших замечаний вкусовое, выданное за существенное? | Нет. Сравнение `hoursPerShift > 10.0` записано как `minor`: для литералов из примера оно точное, но при накопленной погрешности граница может сработать неверно. Это не повод ставить `blocker`. |
| Какое из полученных вами замечаний вы отклонили и на каком основании? | Парное ревью не проводилось, полученных замечаний нет, отклонять было нечего. |

### 7.2 Контрольные вопросы

1. Чем ревью отличается от тестирования и от статического анализа? Что из этого дешевле и почему ни одно не заменяет остальные.
   **Ответ:** Тест проверяет поведение на конкретных данных. Он дешевле отладки на работающей системе, но молчит про ветку, которую не вызвали. Статический анализатор смотрит весь текст без запуска и дёшево ловит шаблон: незакрытый ресурс, заведомо мёртвый код, сравнение строк через `==`. Формулу предметной области он не понимает. Ревью дороже по времени человека и ловит смысл: кто отвечает за расчёт, верно ли правило отбора бригады, можно ли переиспользовать метод. Анализатор не спорит о требованиях, тест не оценивает устройство кода, ревью не перебирает все границы. Поэтому три проверки стоят рядом, а не вместо друг друга.
2. Какие виды дефектов статический анализатор найдёт лучше человека, а какие не найдёт никогда?
   **Ответ:** Лучше человека анализатор находит однообразные дефекты по всему проекту: ресурс без закрытия, возможный `null`, недостижимый код, нарушение соглашений об именах, часть гонок по известному шаблону (`SimpleDateFormat` в поле, доступном нескольким потокам). Он не найдёт ошибку постановки. Что норма простоя — сутки, что бригада с 11 часами не должна попасть в наряд, что отрицательная плата бессмысленна: этого нет в правилах анализатора, это знает только человек, который понимает задачу.
3. Из каких трёх частей состоит полезное замечание? Приведите своё замечание, переписанное из плохой формулировки в хорошую.
   **Ответ:** Полезное замечание состоит из места, последствия и предложения. Говорят о коде, а не об авторе. Плохо: «А если бригад меньше, чем нужно?» Хорошо: «Строки 30–33: после фильтрации в списке может остаться меньше бригад, чем `need`, и `brigades.get(i)` бросит `IndexOutOfBoundsException`. Наряд при этом не формируется, а причина для вызывающего кода выглядит как внутренняя ошибка списка. Предлагаю сравнить `need` с размером до цикла и выбросить `IllegalArgumentException` с текстом, сколько бригад просили и сколько доступно.»
4. Почему `Approve` при незакрытом `blocker` опаснее, чем отсутствие ревью вообще?
   **Ответ:** `Approve` — сигнал остальным, что запрос проверен и его можно сливать. Если при этом в обсуждении висит `blocker`, отметка врёт: дефект формально принят. Отсутствие ревью честнее. Запрос ещё не одобрен, и следующий смотрит его как непроверенный. Ложное одобрение эту осторожность снимает, и ошибка попадает в основную ветку уже с пометкой, что ревью пройдено.
5. Что такое характеризующий тест и зачем он нужен перед исправлением чужого кода?
   **Ответ:** Характеризующий тест записывает нынешнее поведение до правок. Сначала фиксируют ожидаемый результат исходной версии, потом меняют код и смотрят, что разрешённое поведение не уехало. Без этого рефакторинг легко меняет состав наряда, и заметить это нечем. Здесь `ShiftPlanLegacy` на данных из `main` падает с `ConcurrentModificationException` и строку не печатает, поэтому ожидание взято по правилам отбора для корректного случая `need = 2`: остаются Петров и Козлов, строка `Петров; Козлов; `.
6. Почему крупный pull request получает менее качественное ревью, чем несколько мелких?
   **Ответ:** В большом запросе много несвязанных правок. Рецензент устаёт и сползает к вычитке оформления: видит имена и пропускает условие. Мелкий запрос помещается в рабочую память, каждое изменение видно на фоне одной задачи, и замечание получается про конкретную строку. Такой запрос проще вернуть на доработку и проще откатить, если в нём `blocker`.

### 7.3 Три вывода по работе

1. Первый упавший дефект прячет остальные. Пока `make` падает на изменении списка, пустой `catch` и незакрытый файл в выводе не появляются: их находит только чтение кода ниже точки падения.
2. Замечание без места, последствия и серьёзности автору нечего делать. В таблице каждая запись привязана к строкам `ShiftPlanLegacy` и к конкретной правке в `ShiftPlanFixed`.
3. Перед исправлением нужен ожидаемый результат. Иначе правка «заодно» меняет состав наряда, и характеризующий тест как раз не даёт этому пройти незамеченным.


---
## 8. Чек-лист и самопроверка

- [x] Образец изучен, отвечен вопрос п. 3.2
- [x] Таблица замечаний заполнена: не менее восьми записей, все поля непустые
- [x] Не менее трёх замечаний серьёзности `blocker` или `major`
- [x] В каждом замечании указано место, последствие и предложение
- [x] `ShiftPlanFixed.java` написан, самопроверки проходят
- [x] Заполнена таблица прослеживаемости «замечание → изменение»
- [ ] На GitHub оставлено ревью, не менее пяти замечаний, привязанных к строкам (пропущено: парное ревью не делаем)
- [ ] На полученные замечания даны ответы, обсуждения закрыты по существу (пропущено: чужих замечаний нет)
- [x] Отвечены контрольные вопросы и сформулированы выводы
- [ ] Отчёт самопроверки вставлен в описание pull request (после того как запрос будет создан)


In [ ]:
#@title Запустить самопроверку и получить отчёт
import datetime, os, re, subprocess

import pandas as pd

ФИО_исполнителя = "Прач Ксения"  #@param {type:"string"}
Группа_исполнителя = "ГИ-41"  #@param {type:"string"}
ТАБЛИЦА_ПРОСЛЕЖИВАЕМОСТИ_ЗАПОЛНЕНА = True  #@param {type:"boolean"}
ВОПРОСЫ_И_ВЫВОДЫ_ЗАПОЛНЕНЫ = True  #@param {type:"boolean"}
# Парное ревью однокурсника в этой сдаче не выполняется.
ПАРНОЕ_РЕВЬЮ_ПРОПУЩЕНО = True  #@param {type:"boolean"}

ОБЯЗАТЕЛЬНЫЕ_ПОЛЯ = ["Строка", "Категория", "Серьёзность", "Замечание", "Предложение"]
ВЕСОМЫЕ = {"blocker", "major"}
ОБРАЗЕЦ_ССЫЛКИ = re.compile(r"https://github\.com/[\w.-]+/[\w.-]+/pull/\d+")

ошибки = []
строки_отчёта = [
    "### Отчёт самопроверки «Code review»",
    "",
    "| Параметр | Значение |",
    "|---|---|",
    "| Исполнитель | %s, группа %s |" % (ФИО_исполнителя or "НЕ УКАЗАН", Группа_исполнителя or "НЕ УКАЗАНА"),
    "| Дата отчёта (UTC) | %s |" % datetime.datetime.now(datetime.timezone.utc).strftime("%Y-%m-%d %H:%M"),
]

# --- 1. таблица замечаний ---
if not os.path.exists("ReviewNotes.csv"):
    ошибки.append("нет файла ReviewNotes.csv")
    строки_отчёта.append("| Таблица замечаний | ОТСУТСТВУЕТ |")
else:
    таблица = pd.read_csv("ReviewNotes.csv").fillna("")
    отсутствующие_поля = [п for п in ОБЯЗАТЕЛЬНЫЕ_ПОЛЯ if п not in таблица.columns]
    if отсутствующие_поля:
        ошибки.append("в ReviewNotes.csv нет колонок: " + ", ".join(отсутствующие_поля))
        строки_отчёта.append("| Таблица замечаний | нет колонок: %s |" % ", ".join(отсутствующие_поля))
    else:
        маска_заполненных = таблица[ОБЯЗАТЕЛЬНЫЕ_ПОЛЯ].astype(str).apply(
            lambda строка: all(значение.strip() for значение in строка), axis=1)
        заполненные = таблица[маска_заполненных]
        весомых = заполненные["Серьёзность"].astype(str).str.strip().str.lower().isin(ВЕСОМЫЕ).sum()
        коротких = (заполненные["Предложение"].astype(str).str.len() < 20).sum()
        строки_отчёта.append("| Таблица замечаний | записей %d, заполненных %d, весомых %d |"
                             % (len(таблица), len(заполненные), весомых))
        if len(заполненные) < 8:
            ошибки.append("полностью заполненных замечаний %d, требуется не менее 8" % len(заполненные))
        if весомых < 3:
            ошибки.append("замечаний blocker/major %d, требуется не менее 3" % весомых)
        if коротких:
            ошибки.append("замечаний без содержательного предложения: %d" % коротких)

# --- 2. исправленный код ---
def запустить_java(имя_файла):
    if not os.path.exists(имя_файла):
        return False, "файл отсутствует"
    завершение = subprocess.run(["java", имя_файла], capture_output=True, text=True, timeout=180)
    вывод = завершение.stdout + завершение.stderr
    значимые = [с for с in вывод.splitlines() if с.strip() and not с.startswith("Picked up")]
    if завершение.returncode != 0:
        return False, "аварийное завершение: " + (значимые[0][:90] if значимые else "без сообщения")
    if "[FAIL]" in вывод:
        return False, "не пройдено проверок: %d" % вывод.count("[FAIL]")
    успешных = вывод.count("[OK]")
    if успешных < 5:
        return False, "самопроверок в файле только %d, требуется не менее 5" % успешных
    return True, "самопроверок пройдено: %d" % успешных

успех_кода, пояснение_кода = запустить_java("ShiftPlanFixed.java")
строки_отчёта.append("| `ShiftPlanFixed.java` | %s — %s |"
                     % ("пройдено" if успех_кода else "НЕ ПРОЙДЕНО", пояснение_кода))
if not успех_кода:
    ошибки.append("ShiftPlanFixed.java: " + пояснение_кода)

исходный_текст = open("ShiftPlanFixed.java", encoding="utf-8").read() if os.path.exists("ShiftPlanFixed.java") else ""
if исходный_текст and "==" in исходный_текст and re.search(r'==\s*"', исходный_текст):
    ошибки.append("в ShiftPlanFixed.java осталось сравнение строки через ==")

# --- 3. ревью на GitHub ---
ссылка_ревью = globals().get("ССЫЛКА_НА_МОЁ_РЕВЬЮ", "")
ссылка_запроса = globals().get("ССЫЛКА_НА_МОЙ_PR", "")
оставлено = int(globals().get("ЗАМЕЧАНИЙ_ОСТАВЛЕНО", 0) or 0)
получено = int(globals().get("ЗАМЕЧАНИЙ_ПОЛУЧЕНО", 0) or 0)
принято = int(globals().get("ЗАМЕЧАНИЙ_ПРИНЯТО", 0) or 0)

пропустить_ревью = bool(globals().get("ПАРНОЕ_РЕВЬЮ_ПРОПУЩЕНО", False))
if пропустить_ревью:
    строки_отчёта += [
        "| Ревью на чужой запрос | пропущено |",
        "| Свой запрос | пропущено |",
        "| Замечаний оставлено / получено / принято | парное ревью не выполнялось |",
    ]
else:
    строки_отчёта += [
        "| Ревью на чужой запрос | %s |" % (ссылка_ревью or "НЕ УКАЗАНО"),
        "| Свой запрос | %s |" % (ссылка_запроса or "НЕ УКАЗАН"),
        "| Замечаний оставлено / получено / принято | %d / %d / %d |" % (оставлено, получено, принято),
    ]
строки_отчёта += [
    "| Таблица прослеживаемости | %s |" % ("заполнена" if ТАБЛИЦА_ПРОСЛЕЖИВАЕМОСТИ_ЗАПОЛНЕНА else "НЕ ЗАПОЛНЕНА"),
    "| Вопросы и выводы | %s |" % ("заполнены" if ВОПРОСЫ_И_ВЫВОДЫ_ЗАПОЛНЕНЫ else "НЕ ЗАПОЛНЕНЫ"),
]

if not пропустить_ревью:
    for подпись, ссылка in [("ревью", ссылка_ревью), ("своего запроса", ссылка_запроса)]:
        if not ОБРАЗЕЦ_ССЫЛКИ.search(str(ссылка)):
            ошибки.append("ссылка %s не похожа на адрес pull request" % подпись)
    if оставлено < 5:
        ошибки.append("оставлено замечаний %d, требуется не менее 5" % оставлено)
if получено and принято > получено:
    ошибки.append("принято замечаний больше, чем получено")
if not ФИО_исполнителя.strip() or not Группа_исполнителя.strip():
    ошибки.append("не указаны ФИО и группа")
if not ТАБЛИЦА_ПРОСЛЕЖИВАЕМОСТИ_ЗАПОЛНЕНА:
    ошибки.append("не заполнена таблица прослеживаемости")
if not ВОПРОСЫ_И_ВЫВОДЫ_ЗАПОЛНЕНЫ:
    ошибки.append("не заполнены контрольные вопросы и выводы")

print("\n".join(строки_отчёта))
print("\nСкопируйте отчёт выше в описание pull request.\n")

assert not ошибки, "Работа не готова к сдаче: " + "; ".join(ошибки)
print("Самопроверка пройдена: работа готова к сдаче.")
